# Generating prime numbers with the Sieve of Eratosthenes

Prime numbers are integers greater than $1$ whose only positive divisors are $1$ and themselves:

$$2,3,5,7,11,13,17,19,\ldots$$

In this notebook we will generate **all primes up to and including $N$**. We will first see why testing numbers one at a time repeats work, then visualize and implement the Sieve of Eratosthenes.

## 1. A direct approach: test one number at a time

If $n=ab$ is composite, at least one factor is at most $\sqrt n$. Therefore, to test whether one number is prime, it is enough to try divisors from $2$ through $\lfloor\sqrt n\rfloor$. This is reasonable for a single number, but finding every prime up to $N$ repeats many divisibility tests.

In [20]:
from math import isqrt

def is_prime_by_trial_division(n):
    """Return True exactly when the integer n is prime."""
    if n < 2:
        return False
    for d in range(2, isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

[n for n in range(2, 31) if is_prime_by_trial_division(n)]

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

In [21]:
def primes_up_to(n):
    return [k for k in range(2, n) if is_prime_by_trial_division(k)]

In [4]:
primes_up_to(100)

[2,
 3,
 5,
 7,
 11,
 13,
 17,
 19,
 23,
 29,
 31,
 37,
 41,
 43,
 47,
 53,
 59,
 61,
 67,
 71,
 73,
 79,
 83,
 89,
 97]

## 2. The Sieve of Eratosthenes

> **Central idea:** do not test every number separately. Start with all candidates and systematically eliminate the numbers that must be composite.

### The procedure

Begin with every integer from $2$ through $N$:

$$2,3,4,5,6,\ldots,N$$

| Stage | What we discover | What we eliminate |
|:---:|---|---|
| 1 | $2$ is the first survivor, so it is prime. | $4,6,8,10,\ldots$ |
| 2 | $3$ is the next survivor, so it is prime. | $6,9,12,15,\ldots$ |
| 3 | $5$ is the next survivor, so it is prime. | $10,15,20,25,\ldots$ |
| 4 | Continue with $7,11,13,\ldots$ | Their larger multiples |

Numbers such as $6$ or $12$ may be encountered more than once. That is harmless: once crossed out, they stay crossed out.

**What remains?** When the process ends, the surviving numbers are exactly the primes.

The algorithm can be summarized as a loop:

1. Select the smallest number that has not been crossed out.
2. Declare it prime.
3. Cross out its multiples.
4. Repeat with the next survivor.

---

### Why does the sieve work?

Suppose $n$ is composite. Then $n=ab$ for some integers $a,b>1$. At least one factor must be no larger than $\sqrt n$; otherwise,

$$a>\sqrt n \quad\text{and}\quad b>\sqrt n \quad\Longrightarrow\quad ab>n,$$

which is impossible. Therefore every composite $n\le N$ has a prime factor $p$ satisfying

$$p\le\sqrt n\le\sqrt N.$$

> **Stopping rule:** we only need to process primes $p\le\sqrt N$. After that, every remaining candidate is prime.

For $N=100$, we have $\sqrt{100}=10$. Thus only $2,3,5,$ and $7$ need to do the crossing out.

---

### Why start crossing out at $p^2$?

When we reach a prime $p$, every smaller multiple

$$2p,3p,\ldots,(p-1)p$$

has already been eliminated by a smaller prime factor. For example, when $p=5$:

$$10=2\cdot5,\qquad 15=3\cdot5,\qquad 20=4\cdot5.$$

So the first multiple of $p$ that might still be present is $p\cdot p=p^2$.

| Prime $p$ | Begin at $p^2$ | Multiples considered |
|:---:|:---:|---|
| $2$ | $4$ | $4,6,8,10,\ldots$ |
| $3$ | $9$ | $9,12,15,18,\ldots$ |
| $5$ | $25$ | $25,30,35,40,\ldots$ |
| $7$ | $49$ | $49,56,63,70,\ldots$ |

**A useful mental picture:** A composite number survives only until one of its prime factors is processed. A prime survives forever because no smaller prime can eliminate it.


In [22]:
def sieve_steps(N):
    """Record (prime used, newly crossed-out numbers) for a sieve through N."""
    if N < 2:
        return []
    possible = [True] * (N + 1)
    possible[0:2] = [False, False]
    steps = []
    for p in range(2, isqrt(N) + 1):
        if possible[p]:
            removed = []
            for multiple in range(p * p, N + 1, p):
                if possible[multiple]:
                    removed.append(multiple)
                    possible[multiple] = False
            steps.append((p, removed))
    return steps

for p, removed in sieve_steps(100):
    print(f"p = {p}: newly crossed out {removed}")

p = 2: newly crossed out [4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100]
p = 3: newly crossed out [9, 15, 21, 27, 33, 39, 45, 51, 57, 63, 69, 75, 81, 87, 93, 99]
p = 5: newly crossed out [25, 35, 55, 65, 85, 95]
p = 7: newly crossed out [49, 77, 91]


## 3. Watch the sieve

Run the next cell, then click **▶ Play** below the grid. You can also pause or move one step at a time with the controls. Red entries are composite, green entries are confirmed primes, and pale entries are still candidates. The final frame confirms all surviving primes, and the complete list appears below the animation. Choose `N_visual` to be a **perfect square** such as $25$, $49$, $100$, or $144$; the numbers will be arranged in a $\sqrt N\times\sqrt N$ square.

In [23]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

def animate_sieve(N=100, interval=1200):
    """Display a playable animation of the sieve and list the final primes."""
    if N < 2:
        raise ValueError("N must be at least 2 for the visualization.")

    side = isqrt(N)
    if side * side != N:
        raise ValueError("N must be a perfect square, such as 25, 49, 100, or 144.")
    columns = rows = side
    colors = {0: "#fff3bf", 1: "#8ce99a", 2: "orange"}
    state = [0] * (N + 1)  # 0: candidate, 1: prime, 2: composite
    state[0] = state[1] = 2
    frames = [(state.copy(), "Start: every number from 2 to N is a candidate")]

    for p, removed in sieve_steps(N):
        state[p] = 1
        for multiple in removed:
            state[multiple] = 2
        frames.append((state.copy(),
                       f"Select {p}; cross out multiples starting at {p}² = {p*p}"))

    for n in range(2, N + 1):
        if state[n] == 0:
            state[n] = 1
    primes = [n for n in range(2, N + 1) if state[n] == 1]
    frames.append((state.copy(), f"Finished: {len(primes)} primes survive"))

    figure_size = max(3.1, 0.47 * side)
    fig, ax = plt.subplots(figsize=(figure_size, figure_size))
    ax.set(xlim=(-0.5, columns - 0.5), ylim=(rows - 0.5, -0.5), xticks=[], yticks=[])
    ax.set_aspect("equal")
    boxes = []
    for n in range(1, N + 1):
        x, y = (n - 1) % columns, (n - 1) // columns
        boxes.append(ax.text(x, y, str(n), ha="center", va="center", fontsize=11,
                             bbox=dict(boxstyle="round,pad=0.35",
                                       facecolor=colors[0], edgecolor="#555555")))
    title = ax.set_title("", pad=12, fontsize=10)

    def update(frame_number):
        frame_state, frame_title = frames[frame_number]
        for n, box in enumerate(boxes, start=1):
            box.get_bbox_patch().set_facecolor(colors[frame_state[n]])
            box.set_alpha(0.25 if frame_state[n] == 2 else 1.0)
        title.set_text(f"Step {frame_number} of {len(frames)-1} — {frame_title}")
        return boxes + [title]

    animation = FuncAnimation(fig, update, frames=len(frames), interval=interval,
                              repeat=False, blit=False)
    animation_html = animation.to_jshtml(default_mode="once")
    plt.close(fig)
    display(HTML(animation_html))
    print(f"Primes up to {N}:")
    print(primes)
    return primes

N_visual = 400
visual_primes = animate_sieve(N_visual)

Primes up to 400:
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311, 313, 317, 331, 337, 347, 349, 353, 359, 367, 373, 379, 383, 389, 397]


## 4. An efficient implementation

A Boolean NumPy array records whether each integer is still a possible prime. The slice `is_prime[p*p : N+1 : p]` means: start at $p^2$ and visit every $p$-th position. Assigning `False` crosses out all those multiples in one array operation.

In [24]:
import numpy as np

def sieve(N):
    """Return a NumPy array containing every prime p with 2 <= p <= N."""
    if not isinstance(N, (int, np.integer)):
        raise TypeError("N must be an integer.")
    if N < 2:
        return np.array([], dtype=np.int64)

    is_prime = np.ones(N + 1, dtype=bool)
    is_prime[:2] = False
    for p in range(2, isqrt(N) + 1):
        if is_prime[p]:
            is_prime[p * p : N + 1 : p] = False
    return np.flatnonzero(is_prime)

N = 1000000000  # Change this value
primes = sieve(N)
print(f"There are {len(primes)} primes up to {N}.")
print(primes)

There are 50847534 primes up to 1000000000.
[        2         3         5 ... 999999893 999999929 999999937]


## 5. Check the result

Known values provide quick tests: $\pi(10)=4$, $\pi(100)=25$, and $\pi(10^6)=78{,}498$, where $\pi(x)$ counts primes at most $x$. We also test edge cases.

In [25]:
assert sieve(-5).tolist() == []
assert sieve(1).tolist() == []
assert sieve(2).tolist() == [2]
assert sieve(10).tolist() == [2, 3, 5, 7]
assert len(sieve(100)) == 25
assert len(sieve(1_000_000)) == 78_498
print("All checks passed.")

All checks passed.


## 6. How efficient is it?

The sieve uses $O(N)$ Boolean entries and takes $O(N\log\log N)$ time. This is excellent for moderate $N$, but memory matters when $N$ is huge. A billion Boolean entries require about 1 GB in NumPy, before other overhead.

A **segmented sieve** avoids storing the whole interval. It first computes the small primes through $\sqrt N$, then processes blocks such as $[2,M]$, $[M+1,2M]$, and so on. Each block is crossed out using the same small primes and can be discarded after its primes are used or saved. Thus memory depends mainly on the block size, not on $N$.

In [26]:
def segmented_sieve(N, block_size=1_000_000):
    """Return all primes <= N while sieving only one block at a time."""
    if N < 2:
        return np.array([], dtype=np.int64)
    base_primes = sieve(isqrt(N))
    pieces = []
    for low in range(2, N + 1, block_size):
        high = min(low + block_size - 1, N)
        block = np.ones(high - low + 1, dtype=bool)
        for p in base_primes:
            p = int(p)
            start = max(p * p, ((low + p - 1) // p) * p)
            if start > high:
                continue
            block[start - low : high - low + 1 : p] = False
        pieces.append(np.flatnonzero(block) + low)
    return np.concatenate(pieces)

assert np.array_equal(segmented_sieve(100_000, block_size=10_000), sieve(100_000))
print("The ordinary and segmented sieves agree through 100,000.")

The ordinary and segmented sieves agree through 100,000.


## 7. Save a prime dataset (optional)

All primes below $10^9$ number $50{,}847{,}534$; the largest is $999{,}999{,}937$. Holding the returned Python/NumPy result still takes substantial memory, so production-scale generation should stream segmented results to disk or use an optimized program such as `primesieve`. For classroom-sized values, save the array directly:

In [28]:

# Uncomment to save and reload the primes generated in Section 4.
N=1000000000
np.save(f"primes_up_to_{N}.npy", primes.astype(np.uint32))
primes_reloaded = np.load(f"primes_up_to_{N}.npy")
print(primes_reloaded)

[        2         3         5 ... 999999893 999999929 999999937]


In [29]:
primes_reloaded

array([        2,         3,         5, ..., 999999893, 999999929,
       999999937], dtype=uint32)

In [30]:
len(primes_reloaded)

50847534

## Exercises

1. Change `N` and predict the last prime before running the cell.
2. Modify `sieve_steps` to count how many entries are newly crossed out at each stage.
3. Why is $49$ not crossed out until the sieve processes $7$?
4. Compare the running times of trial division and `sieve` for increasing values of $N$.
5. Modify the sieve to represent only odd numbers. How much memory should that save?

In [32]:
primes =primes_reloaded

In [41]:
def describe_distribution(k, N):
    requested = primes[:N]
    residues = {i: [] for i in range(k)}

    for item in requested:
        j = int(item % k)
        residues[j].append(item)

    return residues

In [53]:
ans=describe_distribution(3,1000000)

In [54]:
ans[0]

[np.uint32(3)]

In [55]:
len(ans[1])

499829

In [56]:
len(ans[2])

500170